# KulitKu — percobaan seimbang 100 foto per kelas
Sumber: `MyDrive/KulitKu/datasets/non_aug_v1`. Empat kelas × 100 foto = 400 foto; masing-masing 70 training, 15 validation, 15 test. Jalankan sel berurutan pada Colab (GPU T4 disarankan).

Tiga CNN beku × LR/SVM/RF. Pemilihan model berdasarkan validation, kalibrasi sigmoid melalui 3-fold internal **training saja**, lalu test model terpilih. Grafik skor train–validation dan confusion matrix tersedia. Tidak ada loss per epoch karena CNN tidak di-fine-tune. Log-loss model terkalibrasi dilaporkan per bagian, bukan kurva epoch.

Ini percobaan eksploratori. Nama tanpa `aug_` belum membuktikan foto asli atau pasien berbeda. Audit duplikat piksel dan kandidat kemiripan membantu, tetapi bukan verifikasi pasien. Data ini pernah digunakan pada percobaan sebelumnya; test baru ini bukan test eksternal yang sepenuhnya belum pernah dilihat. Tidak ada klaim diagnosis klinis.

Versi ini tidak memakai 5-fold CV luar, benchmark pemilihan berbasis latensi, atau ekspor ONNX. Kalibrasi mengubah model probabilitas; jangan memakai fungsi ekspor notebook lama langsung pada hasil ini.

## 1. Ambil kode GitHub
Checkout dikunci pada commit yang diperiksa agar perubahan main tidak mengubah percobaan.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
COMMIT = '453e83711490083d76ec53fb7f631b8a665cf946'
CHECKOUT = Path('/content/KulitKu_100_code')
CODE_ROOT = CHECKOUT / 'machine-learning'
if not CHECKOUT.exists():
    subprocess.run(['git','clone','https://github.com/Liex2810/KulitKu.git',str(CHECKOUT)],check=True)
    subprocess.run(['git','-C',str(CHECKOUT),'checkout',COMMIT],check=True)
actual = subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
if actual != COMMIT:
    raise ValueError('Checkout berbeda. Gunakan folder kode terpisah; jangan menimpa pekerjaan.')
print('Kode:', CODE_ROOT, actual)


## 2. Instalasi
Jika diminta restart, restart lalu jalankan lagi dari sel 1 dengan `INSTALL=False`.

In [ ]:
INSTALL = True
if INSTALL:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(CODE_ROOT/'requirements.txt'),'ImageHash==4.3.2'],check=True)
sys.path.insert(0,str(CODE_ROOT/'src'))


## 3. Drive dan pengaturan
Default tanpa augmentasi dahulu agar efek jumlah asli seimbang terlihat. `GROUPS_CSV` opsional berisi `relative_path,group_id`; gunakan ID pasien bila tersedia, atau kelompok foto asal. Hanya satu foto diambil per kelompok, sehingga ukuran tepat 100/kelas dan kelompok tidak melintasi bagian. ID kelompok harus unik global. Jangan menyatakan pasien terverifikasi jika hanya memakai hash.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import hashlib, io, time, warnings
import numpy as np, pandas as pd, torch, joblib, matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash
from IPython.display import display
from kulitku_ml.config import load_config, Run
SEED = 42
DATASET_ROOT = Path('/content/drive/MyDrive/KulitKu/datasets/non_aug_v1')
OUTPUT_ROOT = '/content/drive/MyDrive/KulitKu/hasil_100_per_kelas'
GROUPS_CSV = ''  # opsional: CSV relative_path,group_id
RUN_ID = ''  # isi ID tercetak untuk melanjutkan konfigurasi yang sama
cfg = load_config(CODE_ROOT/'configs/experiment.json')
cfg['seed'] = SEED
cfg['paths'].update(dataset_root=str(DATASET_ROOT),output_root=OUTPUT_ROOT)
for grid in cfg['classifiers'].values():
    grid['model__class_weight'] = ['balanced']
cfg['pilot'] = dict(per_class=100,train=70,validation=15,test=15,
                    augmentation=False,groups_csv=GROUPS_CSV,protocol='holdout_70_15_15_calibration_train_cv3')
run = Run(cfg,RUN_ID or None)
CLASSES = run.c['classes']
print('RUN_ID:',run.id,'\nHasil:',run.out)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 4. Inventaris, decode dan duplikat
Membaca seluruh kandidat non-aug agar pemilihan reproducible. Duplikat piksel RGB dibuang, konflik label menghentikan proses. Sumber tidak disalin/dihapus. CSV kelompok bila dipakai wajib mencakup seluruh kandidat valid.

In [ ]:
rows, errors = [], []
extensions = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff','.jfif'}
for label in CLASSES:
    folder = DATASET_ROOT / label
    if not folder.is_dir(): raise FileNotFoundError(folder)
    for p in sorted(folder.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in extensions or p.name.lower().startswith('aug_'): continue
        try:
            with Image.open(p) as im:
                if im.width*im.height > cfg['cleaning']['max_pixels']: raise ValueError('Melebihi batas piksel')
                rgb = ImageOps.exif_transpose(im).convert('RGB')
                h = hashlib.sha256(str(rgb.size).encode()+rgb.tobytes()).hexdigest()
                rows.append(dict(relative_path=p.relative_to(DATASET_ROOT).as_posix(),label=label,
                    image_id=h,pixel_hash=h,phash=str(imagehash.phash(rgb)),width=rgb.width,height=rgb.height))
        except Exception as e: errors.append(dict(path=str(p),error=str(e)))
    print(label, 'selesai',flush=True)
all_images = pd.DataFrame(rows)
run.table(pd.DataFrame(errors,columns=['path','error']),'unreadable.csv')
if all_images.empty: raise ValueError('Tidak ada gambar valid')
conflicts = all_images.groupby('pixel_hash').label.nunique()
conflicts = all_images[all_images.pixel_hash.isin(conflicts[conflicts>1].index)]
run.table(conflicts,'duplicate_label_conflicts.csv')
if len(conflicts): raise ValueError('Konflik label duplikat: periksa tabel, jangan lanjut.')
run.table(all_images[all_images.duplicated('pixel_hash',keep=False)],'exact_duplicates.csv')
pool = all_images.drop_duplicates('pixel_hash').copy()
if GROUPS_CSV:
    groups = pd.read_csv(GROUPS_CSV,dtype=str).fillna('')
    if groups.relative_path.duplicated().any(): raise ValueError('relative_path ganda pada CSV kelompok')
    pool = pool.merge(groups[['relative_path','group_id']],on='relative_path',how='left',validate='one_to_one')
    if pool.group_id.isna().any() or pool.group_id.eq('').any(): raise ValueError('ID kelompok belum lengkap')
    if pool.groupby('group_id').label.nunique().max()>1: raise ValueError('Kelompok lintas kelas perlu protokol khusus')
else:
    pool['group_id'] = pool.pixel_hash
pool = pool.sort_values('relative_path').drop_duplicates('group_id').reset_index(drop=True)
display(pool.groupby('label').size().rename('kandidat_kelompok_unik'))
if (pool.groupby('label').size().reindex(CLASSES,fill_value=0)<100).any():
    raise ValueError('Setelah cleaning/kelompok, ada kelas kurang dari 100. Jangan menduplikasi untuk memenuhi kuota.')
run.table(pool,'candidate_pool.csv')


## 5. Pilih 100 per kelas dan split
Seed tetap. Jika manifest sudah tersimpan, harus sama dengan hasil yang dihitung ulang. Kandidat mirip antarbagian ditampilkan untuk review, bukan otomatis dianggap duplikat.

In [ ]:
parts=[]
for label in CLASSES:
    sample=pool[pool.label.eq(label)].sample(n=100,random_state=SEED).reset_index(drop=True)
    sample['partition']=['train']*70+['validation']*15+['test']*15
    parts.append(sample)
splits=pd.concat(parts,ignore_index=True)
splits['fold']=-1
old=run.out/'tabel'/'pilot_splits.csv'
if old.exists():
    previous=pd.read_csv(old)
    if previous[['relative_path','image_id','partition','group_id']].astype(str).to_dict('records') != splits[['relative_path','image_id','partition','group_id']].astype(str).to_dict('records'):
        raise ValueError('Manifest berubah. Gunakan run baru.')
assert splits.groupby('group_id').partition.nunique().max()==1
assert splits.pixel_hash.nunique()==400
run.table(splits,'pilot_splits.csv')
summary=pd.crosstab(splits.label,splits.partition).reindex(CLASSES)
run.table(summary.reset_index(),'split_summary.csv'); display(summary)
fig,ax=plt.subplots(figsize=(8,4));summary.plot.bar(ax=ax);ax.set_ylabel('Foto asli');run.figure(fig,'split_counts.png')
pairs=[]
for i in range(len(splits)):
    for j in range(i):
        distance=(int(splits.iloc[i].phash,16)^int(splits.iloc[j].phash,16)).bit_count()
        if distance<=6 and splits.iloc[i].partition!=splits.iloc[j].partition:
            pairs.append(dict(path_a=splits.iloc[i].relative_path,path_b=splits.iloc[j].relative_path,distance=distance))
near=pd.DataFrame(pairs,columns=['path_a','path_b','distance'])
run.table(near,'near_duplicate_review.csv');display(near.head(30))
print('Pasangan kandidat mirip antarbagian:',len(near))


## 6. Review kemiripan
Jika ada pasangan, lihat galeri dan CSV. Jika foto asal/pasien sama, isi CSV kelompok pada sel 3 lalu buat run baru. Jika semua pasangan sudah diperiksa dan memang berbeda, ubah `REVIEWED_DISTINCT=True`. pHash dapat melewatkan crop/duplikat; daftar kosong bukan bukti pasien independen.

In [ ]:
REVIEWED_DISTINCT = False
for row in near.head(12).itertuples():
    print(row.path_a,'|',row.path_b)
    for rel in [row.path_a,row.path_b]:
        with Image.open(DATASET_ROOT/rel) as im:
            preview=ImageOps.exif_transpose(im).convert('RGB');preview.thumbnail((220,220));display(preview)
if len(near) and not REVIEWED_DISTINCT:
    raise ValueError('Review pasangan mirip dahulu; perbaiki kelompok atau tandai sudah diperiksa berbeda.')
run.json(dict(exploratory=True,patient_independence_verified=False,
              grouping='provided_groups' if GROUPS_CSV else 'pixel_hash_only',
              near_pairs=len(near),reviewed_distinct=REVIEWED_DISTINCT),'audit_status.json')


## 7. Ekstraksi tiga CNN beku
Preprocessing sesuai pretrained weights. Seluruh 400 foto boleh diekstrak karena CNN beku; scaler dan classifier hanya di-fit pada training.

In [ ]:
from kulitku_ml.features import extract_features, restore_extractor
for cnn in cfg['extractors']:
    x=extract_features(run,splits,cnn)
    print(cnn,x.shape)
y=splits.label.map({label:i for i,label in enumerate(CLASSES)}).to_numpy()
tr=np.flatnonzero(splits.partition.eq('train'))
va=np.flatnonzero(splits.partition.eq('validation'))
te=np.flatnonzero(splits.partition.eq('test'))


## 8. Training sembilan kombinasi dan pemilihan parameter
Grid berasal dari JSON, bobot balanced sudah diterapkan sebelum Run. Semua kandidat memakai 280 training dan 60 validation yang sama. Test belum digunakan. Model tidak di-refit ke validation agar perbandingan tetap 70/15/15. Ringkasan disimpan per kombinasi untuk melanjutkan sesi.

In [ ]:
from sklearn.model_selection import ParameterGrid
from kulitku_ml.training import classifier, metrics
from sklearn.exceptions import ConvergenceWarning
records=[]
for cnn in cfg['extractors']:
    x=np.load(run.out/'features'/cnn/'features.npy',mmap_mode='r')
    for name,grid in cfg['classifiers'].items():
        combo=cnn+'__'+name
        folder=run.out/'pilot_models'/combo;folder.mkdir(parents=True,exist_ok=True)
        summary_path=folder/'summary.json'
        if summary_path.exists():
            records.append(json.loads(summary_path.read_text()));print('Cache:',combo);continue
        candidates=[]
        for k,params in enumerate(ParameterGrid(grid)):
            print(combo,'kandidat',k+1,flush=True)
            estimator=classifier(run,name,params)
            with warnings.catch_warnings():
                warnings.simplefilter('error',ConvergenceWarning)
                estimator.fit(x[tr],y[tr])
            train_m=metrics(y[tr],estimator.predict(x[tr]));val_m=metrics(y[va],estimator.predict(x[va]))
            candidates.append(dict(candidate=k,params=params,train_macro_f1=train_m['macro_f1'],
                validation_macro_f1=val_m['macro_f1'],train_accuracy=train_m['accuracy'],validation_accuracy=val_m['accuracy']))
        run.table(pd.DataFrame(candidates),f'grid_{combo}.csv')
        winner=sorted(candidates,key=lambda r:(-r['validation_macro_f1'],r['candidate']))[0]
        estimator=classifier(run,name,winner['params']);estimator.fit(x[tr],y[tr])
        joblib.dump(estimator,folder/'base.joblib')
        result=dict(combo_id=combo,cnn=cnn,classifier=name,**winner)
        run.json(result,f'pilot_models/{combo}/summary.json');records.append(result)
results=pd.DataFrame(records).sort_values(['validation_macro_f1','combo_id'],ascending=[False,True])
run.table(results,'train_validation_overview.csv');display(results)
selected=results.iloc[0].to_dict()
lock=run.out/'pilot_selection.json'
if lock.exists() and json.loads(lock.read_text())['combo_id']!=selected['combo_id']:
    raise ValueError('Pilihan terkunci berbeda; gunakan run baru')
run.json(selected,'pilot_selection.json')
print('Terpilih berdasarkan validation:',selected['combo_id'])


## 9. Grafik gap dan confusion matrix train/validation
Gap adalah indikator, tidak otomatis menyimpulkan overfit. Validation dipakai memilih model sehingga skornya dapat optimistis.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, log_loss
plot=results.set_index('combo_id')[['train_macro_f1','validation_macro_f1']]
fig,ax=plt.subplots(figsize=(11,5));plot.plot.bar(ax=ax);ax.set_ylim(0,1.05);ax.set_ylabel('Macro-F1');run.figure(fig,'train_validation_macro_f1.png')
for row in results.itertuples():
    x=np.load(run.out/'features'/row.cnn/'features.npy',mmap_mode='r')
    estimator=joblib.load(run.out/'pilot_models'/row.combo_id/'base.joblib')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,idx,title in [(axes[0],tr,'Training'),(axes[1],va,'Validation')]:
        pred=estimator.predict(x[idx])
        ConfusionMatrixDisplay.from_predictions(y[idx],pred,labels=range(4),display_labels=CLASSES,ax=ax,colorbar=False,xticks_rotation=30)
        ax.set_title(title)
        report=pd.DataFrame(classification_report(y[idx],pred,labels=range(4),target_names=CLASSES,output_dict=True,zero_division=0)).T
        run.table(report.reset_index(names='class'),f'{row.combo_id}_{title.lower()}_report.csv')
    fig.suptitle(row.combo_id);run.figure(fig,row.combo_id+'_train_validation_cm.png')


## 10. Kalibrasi confidence — hanya training
Kalibrasi sigmoid dengan 3-fold berdasarkan kelompok pada 280 training; scaler ikut di-fit ulang di tiap fold. Validation dan test tidak masuk fit kalibrasi. Pemilihan hyperparameter telah memakai validation, jadi skor validation bukan bukti independen kualitas probabilitas. Sampel kecil membuat probabilitas tetap perlu dinilai hati-hati.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone
selected=json.loads((run.out/'pilot_selection.json').read_text())
cnn=selected['cnn'];combo=selected['combo_id']
x=np.load(run.out/'features'/cnn/'features.npy',mmap_mode='r')
base=joblib.load(run.out/'pilot_models'/combo/'base.joblib')
cv=list(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED).split(x[tr],y[tr],groups=splits.iloc[tr].group_id))
for fit_idx,cal_idx in cv:
    assert set(splits.iloc[tr[fit_idx]].group_id).isdisjoint(splits.iloc[tr[cal_idx]].group_id)
    assert len(np.unique(y[tr[fit_idx]]))==len(np.unique(y[tr[cal_idx]]))==4
calibrated_path=run.out/'pilot_models'/combo/'calibrated.joblib'
if calibrated_path.exists(): calibrated=joblib.load(calibrated_path)
else:
    calibrated=CalibratedClassifierCV(estimator=clone(base),method='sigmoid',cv=cv,ensemble=True,n_jobs=1)
    calibrated.fit(x[tr],y[tr]);joblib.dump(calibrated,calibrated_path)
print('Kalibrasi selesai:',combo)


## 11. Test final model terkalibrasi
Hanya model terpilih diuji. Jangan memilih ulang berdasarkan test. Prediksi probabilitas, macro-F1, accuracy, log-loss dan multiclass Brier score disimpan. Log-loss ini bukan loss per epoch. Training merupakan skor pada data yang ikut membangun ensemble sehingga bukan evaluasi independen. Dengan 15 test/kelas, satu kesalahan mengubah recall kelas 6,67 poin persentase.

In [ ]:
final_rows=[]
for part,idx in [('train',tr),('validation',va),('test',te)]:
    prob=calibrated.predict_proba(x[idx]);pred=calibrated.classes_[prob.argmax(axis=1)]
    onehot=(y[idx,None]==calibrated.classes_[None,:]).astype(float)
    final_rows.append(dict(partition=part,**metrics(y[idx],pred),
        log_loss=log_loss(y[idx],prob,labels=calibrated.classes_),multiclass_brier=float(np.mean(np.sum((prob-onehot)**2,axis=1)))))
    predictions=splits.iloc[idx][['image_id','relative_path','label','group_id']].copy()
    predictions['predicted_label']=[CLASSES[int(p)] for p in pred]
    for col,cls in enumerate(calibrated.classes_): predictions['prob_'+CLASSES[int(cls)]]=prob[:,col]
    run.table(predictions,f'calibrated_{part}_predictions.csv')
    fig,ax=plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(y[idx],pred,labels=range(4),display_labels=CLASSES,ax=ax,colorbar=False,xticks_rotation=30)
    ax.set_title('Calibrated '+part);run.figure(fig,f'calibrated_{part}_cm.png')
    report=pd.DataFrame(classification_report(y[idx],pred,labels=range(4),target_names=CLASSES,output_dict=True,zero_division=0)).T
    run.table(report.reset_index(names='class'),f'calibrated_{part}_report.csv')
final=pd.DataFrame(final_rows);run.table(final,'calibrated_metrics.csv');display(final)
fig,ax=plt.subplots(figsize=(7,4));final.set_index('partition')[['macro_f1','accuracy']].plot.bar(ax=ax);ax.set_ylim(0,1.05);run.figure(fig,'calibrated_scores.png')
print('Hasil:',run.out)


## 12. Upload gambar sendiri + confidence persen
Probabilitas model bukan kepastian diagnosis. Model selalu memilih salah satu dari empat kelas, termasuk untuk foto di luar distribusi. Jangan mengubah model berdasarkan foto test lalu melaporkan skor test lama sebagai evaluasi baru.

In [ ]:
from google.colab import files
from kulitku_ml.preprocessing import rgb,build_transform
model,info=restore_extractor(run,cnn);model=model.cpu().eval()
transform=build_transform(info['preprocessing'])
upload_dir=Path('/content/kulitku_pilot_upload');upload_dir.mkdir(exist_ok=True)
for i,(filename,content) in enumerate(files.upload().items()):
    path=upload_dir/f'{i}_{Path(filename).name}';path.write_bytes(content)
    try:
        start=time.perf_counter();image=rgb(path,cfg['cleaning']['max_pixels'])
        with torch.inference_mode(): feature=model(transform(image).unsqueeze(0)).cpu().numpy().astype('float32')
        probabilities=calibrated.predict_proba(feature)[0]
        ranking=pd.DataFrame({'kelas':[CLASSES[int(c)] for c in calibrated.classes_],'probabilitas_persen':probabilities*100}).sort_values('probabilitas_persen',ascending=False)
        elapsed=(time.perf_counter()-start)*1000
        preview=image.copy();preview.thumbnail((500,500));display(preview)
        print(filename,'\nPrediksi:',ranking.iloc[0]['kelas'])
        print(f"Confidence model: {ranking.iloc[0]['probabilitas_persen']:.2f}% | {elapsed:.1f} ms")
        display(ranking.reset_index(drop=True))
    except Exception as e: print('Gagal:',filename,str(e))
